In [ ]:
code = 'SRE_PS_W_Straddle_PSL'
pickle_path = 'C:/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output/'

from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [ ]:
def SREW_SHIFT(bt, start_time, end_time, orderside, method, om, divider, movement1, movement2):
    try:
        start_dt = datetime.datetime.combine(bt.current_week_dates[0], start_time)
        end_dt = datetime.datetime.combine(bt.current_week_dates[-1], end_time)
        end_dt_1m = end_dt + datetime.timedelta(minutes=10)

        entry_time = ''
        from_candle_close = True if method == 'CC' else False

        ce_scrip, pe_scrip, ce_price, pe_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om)
        if ce_scrip is None: return None

        if get_strike(ce_scrip) <= get_strike(pe_scrip):
            start_dt = datetime.datetime.combine(bt.current_week_dates[0], start_time)
            movement = movement2
            check_non_std_trades = False
        else:
            entry_time = start_dt
            movement = movement1
            check_non_std_trades = True

            premium = ce_price + pe_price
            ce_start_dt, pe_start_dt = start_dt, start_dt

        ## Check Non Straddle Trades ##
        trades = []
        shifting_pnl, eod_ce_pnl, eod_pe_pnl, ps_total_pnl = 0, 0, 0, 0
        exit_time = end_dt

        if check_non_std_trades:
            for re in range(max_re+1):

                divider_price = cal_percent(premium, divider)
                move_price = cal_percent(premium, movement)

                start_dt = max(ce_start_dt, pe_start_dt)
                t_ce, t_pe = bt.get_straddle_data(start_dt, end_dt, ce_scrip, pe_scrip, seperate=True)
                ce_data, pe_data = t_ce.copy(), t_pe.copy()

                _, _, _, _, ce_sl_time, _ = bt.sl_check_by_given_data(ce_data, sl_price=move_price, orderside=orderside, from_candle_close=from_candle_close)
                _, _, _, _, pe_sl_time, _ = bt.sl_check_by_given_data(pe_data, sl_price=move_price, orderside=orderside, from_candle_close=from_candle_close)

                ce_sl_time = ce_sl_time if ce_sl_time else end_dt_1m
                pe_sl_time = pe_sl_time if pe_sl_time else end_dt_1m

                if ce_sl_time < pe_sl_time:

                    ce_price_at_sl = bt.options_data.loc[(ce_sl_time, ce_scrip), 'close']
                    pe_price_at_sl = bt.options_data.loc[(ce_sl_time, pe_scrip), 'close']
                    
                    _, shift_mtm_data = bt.sl_check_single_leg(pe_start_dt, ce_sl_time, pe_scrip, per_minute_mtm=True)
                    shift_mtm_data = set_pm_time_index(shift_mtm_data, time_index)
                    trades.append(shift_mtm_data)

                    target_price = pe_price_at_sl + divider_price
                    pe_scrip, pe_price, _, pe_start_dt = bt.get_strike(ce_sl_time, end_dt, target=target_price, obove_target_only=True, only='PE')
                    if pe_scrip is None:
                        exit_time = ce_sl_time
                        break
                    else:
                        if get_strike(ce_scrip) > get_strike(pe_scrip):
                            premium = premium - pe_price_at_sl + pe_price
                        else:
                            _, shift_mtm_data = bt.sl_check_single_leg(ce_start_dt, ce_sl_time, ce_scrip, per_minute_mtm=True)
                            shift_mtm_data = set_pm_time_index(shift_mtm_data, time_index)
                            trades.append(shift_mtm_data)
                            
                            start_dt = ce_sl_time
                            movement = movement2
                            check_non_std_trades = False
                            ce_scrip, pe_scrip = None, None
                            ce_price, pe_price = '', ''
                            break

                elif pe_sl_time < ce_sl_time:

                    ce_price_at_sl = bt.options_data.loc[(pe_sl_time, ce_scrip), 'close']
                    pe_price_at_sl = bt.options_data.loc[(pe_sl_time, pe_scrip), 'close']
                    
                    _, shift_mtm_data = bt.sl_check_single_leg(ce_start_dt, pe_sl_time, ce_scrip, per_minute_mtm=True)
                    shift_mtm_data = set_pm_time_index(shift_mtm_data, time_index)
                    trades.append(shift_mtm_data)

                    target_price = ce_price_at_sl + divider_price
                    ce_scrip, ce_price, _, ce_start_dt = bt.get_strike(pe_sl_time, end_dt, target=target_price, obove_target_only=True, only='CE')
                    if ce_scrip is None: 
                        exit_time = pe_sl_time
                        break
                    else:
                        if get_strike(ce_scrip) > get_strike(pe_scrip):
                            premium = premium - ce_price_at_sl + ce_price
                        else:
                            _, shift_mtm_data = bt.sl_check_single_leg(pe_start_dt, pe_sl_time, pe_scrip, per_minute_mtm=True)
                            shift_mtm_data = set_pm_time_index(shift_mtm_data, time_index)
                            trades.append(shift_mtm_data)

                            start_dt = pe_sl_time
                            movement = movement2
                            check_non_std_trades = False
                            ce_scrip, pe_scrip = None, None
                            ce_price, pe_price = '', ''
                            break

                else:
                    if ce_sl_time != end_dt_1m:

                        ce_price_at_sl = bt.options_data.loc[(ce_sl_time, ce_scrip), 'close']
                        pe_price_at_sl = bt.options_data.loc[(ce_sl_time, pe_scrip), 'close']
                        
                        _, shift_mtm_data = bt.sl_check_single_leg(pe_start_dt, ce_sl_time, pe_scrip, per_minute_mtm=True)
                        shift_mtm_data = set_pm_time_index(shift_mtm_data, time_index)
                        trades.append(shift_mtm_data)

                        _, shift_mtm_data = bt.sl_check_single_leg(ce_start_dt, pe_sl_time, ce_scrip, per_minute_mtm=True)
                        shift_mtm_data = set_pm_time_index(shift_mtm_data, time_index)
                        trades.append(shift_mtm_data)

                        target_price = pe_price_at_sl + divider_price
                        pe_scrip, pe_price, _, pe_start_dt = bt.get_strike(ce_sl_time, end_dt, target=target_price, obove_target_only=True, only='PE')
                        if pe_scrip is None:
                            ce_scrip, exit_time = None, None
                            break

                        target_price = ce_price_at_sl + divider_price
                        ce_scrip, ce_price, _, ce_start_dt = bt.get_strike(ce_sl_time, end_dt, target=target_price, obove_target_only=True, only='CE')
                        if ce_scrip is None:
                            pe_scrip, exit_time = None, None
                            break

                        if get_strike(ce_scrip) > get_strike(pe_scrip):
                            premium = premium - ce_price_at_sl + ce_price - pe_price_at_sl + pe_price
                        else:
                            start_dt = ce_sl_time
                            movement = movement2
                            check_non_std_trades = False
                            ce_scrip, pe_scrip = None, None
                            ce_price, pe_price = '', ''
                            break

                    else:
                        ce_sl_time = end_dt
                        _, shift_mtm_data = bt.sl_check_single_leg(pe_start_dt, ce_sl_time, pe_scrip, per_minute_mtm=True)
                        shift_mtm_data = set_pm_time_index(shift_mtm_data, time_index)
                        trades.append(shift_mtm_data)

                        pe_sl_time = end_dt
                        _, shift_mtm_data = bt.sl_check_single_leg(ce_start_dt, pe_sl_time, ce_scrip, per_minute_mtm=True)
                        shift_mtm_data = set_pm_time_index(shift_mtm_data, time_index)
                        trades.append(shift_mtm_data)

                        ce_scrip, pe_scrip = None, None
                        ce_price, pe_price = '', ''
                        break

            if ce_scrip is None and pe_scrip is not None:
                _, shift_mtm_data = bt.sl_check_single_leg(pe_start_dt, exit_time, pe_scrip, per_minute_mtm=True)
                shift_mtm_data = set_pm_time_index(shift_mtm_data, time_index)
                trades.append(shift_mtm_data)
               
            elif ce_scrip is not None and pe_scrip is None:
                _, shift_mtm_data = bt.sl_check_single_leg(ce_start_dt, exit_time, ce_scrip, per_minute_mtm=True)
                shift_mtm_data = set_pm_time_index(shift_mtm_data, time_index)
                trades.append(shift_mtm_data)

        ## Check Straddle Trades ##
        shifting_pnl, eod_ce_pnl, eod_pe_pnl, total_pnl = 0, 0, 0, 0
        exit_time = end_dt

        if not check_non_std_trades:
            for re in range(std_re_entries+1):
                ce_scrip, pe_scrip, ce_price, pe_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=0)
                if ce_scrip is None: return None

                entry_time = entry_time if entry_time else start_dt

                premium = ce_price + pe_price
                move_price = cal_percent(premium, movement)

                t_ce, t_pe = bt.get_straddle_data(start_dt, end_dt, ce_scrip, pe_scrip, seperate=True)
                ce_data, pe_data = t_ce.copy(), t_pe.copy()

                _, _, _, _, ce_sl_time, _ = bt.sl_check_by_given_data(ce_data, sl_price=move_price, orderside=orderside, from_candle_close=from_candle_close)
                _, _, _, _, pe_sl_time, _ = bt.sl_check_by_given_data(pe_data, sl_price=move_price, orderside=orderside, from_candle_close=from_candle_close)

                ce_sl_time = ce_sl_time if ce_sl_time else end_dt_1m
                pe_sl_time = pe_sl_time if pe_sl_time else end_dt_1m

                if ce_sl_time < pe_sl_time:
                    
                    _, shift_mtm_data = bt.sl_check_single_leg(start_dt, ce_sl_time, pe_scrip, per_minute_mtm=True)
                    shift_mtm_data = set_pm_time_index(shift_mtm_data, time_index)
                    trades.append(shift_mtm_data)
                    
                    _, shift_mtm_data = bt.sl_check_single_leg(start_dt, ce_sl_time, ce_scrip, per_minute_mtm=True)
                    shift_mtm_data = set_pm_time_index(shift_mtm_data, time_index)
                    trades.append(shift_mtm_data)

                    start_dt = ce_sl_time

                elif pe_sl_time < ce_sl_time:
                    
                    _, shift_mtm_data = bt.sl_check_single_leg(start_dt, pe_sl_time, pe_scrip, per_minute_mtm=True)
                    shift_mtm_data = set_pm_time_index(shift_mtm_data, time_index)
                    trades.append(shift_mtm_data)
                    
                    _, shift_mtm_data = bt.sl_check_single_leg(start_dt, pe_sl_time, ce_scrip, per_minute_mtm=True)
                    shift_mtm_data = set_pm_time_index(shift_mtm_data, time_index)
                    trades.append(shift_mtm_data)

                    start_dt = pe_sl_time

                else:
                    if ce_sl_time != end_dt_1m:
                        
                        _, shift_mtm_data = bt.sl_check_single_leg(start_dt, ce_sl_time, ce_scrip, per_minute_mtm=True)
                        shift_mtm_data = set_pm_time_index(shift_mtm_data, time_index)
                        trades.append(shift_mtm_data)
                        
                        _, shift_mtm_data = bt.sl_check_single_leg(start_dt, pe_sl_time, pe_scrip, per_minute_mtm=True)
                        shift_mtm_data = set_pm_time_index(shift_mtm_data, time_index)
                        trades.append(shift_mtm_data)

                        start_dt = ce_sl_time

                    else:
                        ce_sl_time = end_dt
                        _, shift_mtm_data = bt.sl_check_single_leg(start_dt, ce_sl_time, ce_scrip, per_minute_mtm=True)
                        shift_mtm_data = set_pm_time_index(shift_mtm_data, time_index)
                        trades.append(shift_mtm_data)
                        
                        pe_sl_time = end_dt
                        _, shift_mtm_data = bt.sl_check_single_leg(start_dt, pe_sl_time, pe_scrip, per_minute_mtm=True)
                        shift_mtm_data = set_pm_time_index(shift_mtm_data, time_index)
                        trades.append(shift_mtm_data)
                        
                        break

        final_mtm_data = np.sum(trades, axis=0)
        return final_mtm_data

    except Exception as e:
        print(e, [bt.index, bt.current_week_dates[0].date(), bt.current_week_dates[-1].date(), start_time, end_time, orderside, method, om, divider, movement1, movement2])
        return

In [ ]:
def SREW_SHIFT_PSL(bt, start_time, end_time, orderside, method, om, divider, movement1, movement2):
    try:
        start_dt = datetime.datetime.combine(bt.current_week_dates[0], start_time)
        entry_time = start_dt

        per_minute_mtm = SREW_SHIFT(bt, start_time, end_time, orderside, method, om, divider, movement1, movement2)
        if per_minute_mtm is None: return

        if only_mtm:
            mtm_time_list = list(per_minute_mtm)
            mtm_time_list += [mtm_time_list[-1]]
        else:
            per_minute_mtm_total = per_minute_mtm * total_shares

            mtm_dict = {}
            for mtm_percent, check_mtm in check_mtms.items():

                if check_mtm > 0:
                    condition = np.where(per_minute_mtm_total > check_mtm)[0]
                else:
                    condition = np.where(per_minute_mtm_total < check_mtm)[0]

                try:
                    idx = condition[0]
                    time = time_index[idx]
                    mtm = per_minute_mtm_total[idx]
                except:
                    time, mtm = '', per_minute_mtm_total[-1]

                mtm_dict[mtm_percent] = (time, mtm)

            mtm_time_list = [fund] + [item for value in mtm_dict.values() for item in value]
       
        return [code, bt.index, start_time, end_time, orderside, method, om, divider, movement1, movement2, bt.current_week_dates[0].date(), bt.current_week_dates[-1].date(), bt.from_dte, bt.to_dte, len(bt.current_week_dates), entry_time.time(), future_price] + mtm_time_list
    except Exception as e:
        print(e, [bt.index, bt.current_week_dates[0].date(), bt.current_week_dates[-1].date(), start_time, end_time, orderside, method, om, divider, movement1, movement2])
        return

In [ ]:
for row_idx in range(len(meta_data)):
    
    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, from_dte, to_dte, from_date, to_date, start_time, end_time, week_lists = get_meta_row_data(meta_row, pickle_path, weekly=True)
            max_re, re_entries, std_re_entries = 20, 20, 20

            log_colsb = ('P_Strategy/P_Index/P_StartTime/P_EndTime/P_OrderSide/P_Method/P_OM/P_Divider/P_Movement1/P_Movement2/Start.Date/End.Date/Start.DTE/End.DTE/DayCount/EntryTime/Future/')

            only_mtm = True
            if not only_mtm:
                notinal_value = 14
                fund = 100_00_00_00
                mtm_percent_stop = [-0.1, -0.2, -0.3, -0.4, -0.5, -0.6, -0.7, -0.8, -0.9, -1, -1.25, -1.5, -1.75, -2, -2.5, -3, -3.5, -4, -100, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1, 1.25, 1.5, 1.75, 2, 2.5, 3, 3.5, 4]
                log_cols = log_colsb + 'Fund'
                for mtmp in mtm_percent_stop:
                    log_cols += f'/{mtmp:.2f}.Time/{mtmp:.2f}.PNL'
                log_cols = log_cols.split('/')

            for week_dates in week_lists:
                from_date = week_dates[0]
                to_date = week_dates[-1]

                file_name = f"{index} {week_dates[0].date()} {week_dates[-1].date()} {from_dte}-{to_dte} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):

                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")

                    wbt = WeeklyBacktest(pickle_path, index, week_dates, from_dte, to_dte, start_time, end_time)
                    time_index = get_pm_time_index(wbt.current_week_dates, wbt.meta_start_time, wbt.meta_end_time)
                    future_price = wbt.future_data['close'].iloc[0]
                    
                    if not only_mtm:
                        margin_per_share = future_price * (notinal_value / 100)
                        total_shares = fund/margin_per_share
                        check_mtms = {mtm_percent: fund * mtm_percent / 100 for mtm_percent in mtm_percent_stop}
                    else:
                        log_time_col = get_pm_time_index(wbt.current_week_dates, start_time, end_time)
                        log_cols = log_colsb + '/'.join(map(str, log_time_col)) + '/Final.PNL'
                        log_cols = log_cols.split('/')

                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)

                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [SREW_SHIFT_PSL(wbt, row.entry_time, row.exit_time, row.orderside, row.method, row.om, row.divider, row.movement1, row.movement2) for row in tqdm(chunk_parameter.itertuples(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)

                        del chunk
                        del chunk_parameter
                        gc.collect()

                    del wbt
                    gc.collect()

                    t2 = datetime.datetime.now()
                    print(t2-t1)

        except Exception as e:
            input(str(e))